<a href="https://colab.research.google.com/github/VeraZhang0311/cis6280-final-project/blob/emoo-a4-32-hold/cis6280_a4_32_hold_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# A4-32 HOLD optimization — Persistent Colab

This notebook runs a controlled comparison among **A4-32**, the failed long-horizon **A4-32-HOLD**, and the new **A4-32-LOCAL-HOLD**:

- all groups use the same 120k EMA goal actor, frozen world model, 32 candidates, and 16-block rollout;
- both HOLD modes are exactly A4-32 before two real final-goal confirmations;
- old HOLD minimizes blocks 13–16 and stays latched;
- LOCAL-HOLD requires four locally safe blocks, minimizes first-block wrist motion, and exits after two real-observation misses;
- no planner reads simulator box counts, reward, or success.

Switch to a GPU runtime, then choose **Runtime → Run all**. The default is a full-horizon screen on scene `100038`, seed `0`. Results and videos persist under `MyDrive/cis6280-final-project-v2/outputs/a4-32-local-hold/`.


In [ ]:
import os, subprocess, sys
from pathlib import Path

os.environ["MUJOCO_GL"] = "egl"
os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["HF_HUB_DISABLE_XET"] = "1"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_ROOT = Path("/content/drive/MyDrive/cis6280-final-project-v2")
else:
    PERSIST_ROOT = Path("/tmp/cis6280-final-project-v2")
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    env = dict(os.environ, DEBIAN_FRONTEND="noninteractive")
    subprocess.check_call(["apt-get", "update", "-qq"], env=env)
    subprocess.check_call(
        ["apt-get", "install", "-y", "-qq", "libegl1", "libgl1", "libgles2", "libglib2.0-0", "ffmpeg", "git"],
        env=env,
    )

required = {
    "mujoco": "mujoco>=3.3,<4",
    "imageio_ffmpeg": "imageio[ffmpeg]>=2.30",
    "huggingface_hub": "huggingface_hub>=0.26",
    "matplotlib": "matplotlib",
}
missing = []
for module, package in required.items():
    try:
        __import__(module)
    except ImportError:
        missing.append(package)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU is visible. Choose Runtime → Change runtime type → GPU, then Run all.")
print("persistent root:", PERSIST_ROOT)
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPOSITORY = "https://github.com/VeraZhang0311/cis6280-final-project.git"
V2_REF = "origin/emoo-a4-32-hold"
ROOT = PERSIST_ROOT / "source-v2"

if not (ROOT / ".git").is_dir():
    subprocess.check_call(["git", "clone", REPOSITORY, str(ROOT)])
subprocess.check_call(["git", "-C", str(ROOT), "fetch", "origin", "--tags", "--prune"])
dirty = subprocess.check_output(
    ["git", "-C", str(ROOT), "status", "--porcelain"], text=True
).strip()
if dirty:
    raise RuntimeError(f"Persistent source checkout has local changes; clean {ROOT} before continuing:\n{dirty}")
subprocess.check_call(["git", "-C", str(ROOT), "checkout", "--detach", V2_REF])
commit = subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True
).strip()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

hand = ROOT / "assets" / "sharpa" / "wave_01" / "right_sharpa_wave" / "right_sharpa_wave.xml"
if not hand.is_file():
    subprocess.check_call([sys.executable, "scripts/fetch_assets.py"])

from assignment.training.data.task import StackTaskEnv, TaskConfig
from assignment.training.planning.goal_actor import GoalActorAgent

env = StackTaskEnv(TaskConfig(target_count=4, horizon=10, terminate_on_ground=False), 100038)
env.reset(("init", 100038))
env.env.resolution = 128
frame = env.env.render()[0]
env.close()
if frame.shape != (128, 128, 3):
    raise RuntimeError(f"Unexpected render shape: {frame.shape}")
print("v2 code commit:", commit)
print("render smoke test:", frame.shape)


In [ ]:
import hashlib, shutil, urllib.request
from pathlib import Path
from huggingface_hub import hf_hub_download

CACHE = PERSIST_ROOT / "checkpoints"
CACHE.mkdir(parents=True, exist_ok=True)
ACTOR = CACHE / "goal-actor-a4-120k-ema.pt"
WM = CACHE / "jepa-smwm-c3-model.pt"
OFFICIAL_ACTOR = CACHE / "df-smwm-c1-sigma04-policy.pt"
GOALS = CACHE / "scripted-val100-every80-goals.npz"

EXPECTED = {
    ACTOR: "755e95f0fac13fd8990436432986584ce0f8977a23eb5c8d946546fb55477ce3",
    WM: "26590118c9cb38e6af88248a3379c6622c89160d605d68e7d48f586527e821aa",
    OFFICIAL_ACTOR: "8e1e0cea81ab55d0b2223ea28fa70c429227522f2543b98c43f3ae05048788f3",
    GOALS: "acc0ae132bd7ff3566235c03b5302073aade88cf11c41e0b201dca6b6e72dfde",
}

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def valid(path):
    return path.is_file() and sha256(path) == EXPECTED[path]

def install(source, destination):
    if valid(destination):
        print("verified cached:", destination.name)
        return
    if destination.exists():
        raise RuntimeError(f"Checksum mismatch for existing file: {destination}. Remove it manually and rerun.")
    temporary = destination.with_suffix(destination.suffix + ".part")
    if temporary.exists():
        temporary.unlink()
    if str(source).startswith("https://"):
        urllib.request.urlretrieve(source, temporary)
    else:
        shutil.copyfile(source, temporary)
    actual = sha256(temporary)
    if actual != EXPECTED[destination]:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(f"Checksum mismatch for {destination.name}: {actual}")
    temporary.replace(destination)
    print("downloaded and verified:", destination.name)

install(
    "https://github.com/VeraZhang0311/cis6280-final-project/releases/download/v2.0-a4-120k/goal-actor-a4-120k-ema.pt",
    ACTOR,
)
install(hf_hub_download("YongYong/cis6280-stack-world-models-v2", "jepa/smwm-c3/model.pt"), WM)
install(hf_hub_download("YongYong/cis6280-stack-world-models-v2", "policy/df-smwm-c1-sigma04/policy.pt"), OFFICIAL_ACTOR)
install(hf_hub_download("YongYong/cis6280-stack-datasets-v2", "goals/scripted-val100-every80/goals.npz", repo_type="dataset"), GOALS)


## Preflight

This cell validates the exact v2 checkpoint type, the selected cumulative training step, the official dense-goal structure, and the requested scene before the expensive rollout starts.


In [ ]:
import numpy as np
from assignment.training.training.goal_policy import load_goal_policy, validate_a4_checkpoint

actor, actor_info = load_goal_policy(ACTOR, "cpu")
validate_a4_checkpoint(actor_info, 120000)
if actor.cfg.context != 1 or actor.cfg.horizon != 16:
    raise RuntimeError(f"Unexpected actor configuration: {actor.cfg}")
del actor

SCENE = 100038
SAMPLE_SEED = 0
with np.load(GOALS) as goals:
    required = {"seeds", "sub_frames", "sub_count", "init_frames"}
    if not required <= set(goals.files):
        raise RuntimeError(f"Official goals missing fields: {required - set(goals.files)}")
    matches = np.flatnonzero(goals["seeds"] == SCENE)
    if len(matches) != 1:
        raise RuntimeError(f"Scene {SCENE} is not uniquely present in official goals")
    goal_count = int(goals["sub_count"][matches[0]])
print("A4 actor validated at cumulative step:", actor_info["cumulative_step"])
print("scene:", SCENE, "sample seed:", SAMPLE_SEED, "subgoals:", goal_count)


## Controlled LOCAL-HOLD experiment

The default screen runs one paired scene to test whether short-term safety and low wrist intervention retain four boxes longer than both A4-32 and the failed late-four HOLD. LOCAL-HOLD exits after two real observations outside the final-goal threshold so it can resume stacking.

The output includes the immutable manifest, step traces, task counts, summaries, paired results, videos, latch/exit counts, safe-candidate rate, and selected wrist RMS. A one-scene result is debugging evidence only; do not run the frozen 20-scene × 3-seed plan unless this mechanism behaves as intended.


In [ ]:
import json, subprocess, sys
from datetime import datetime, timezone

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = PERSIST_ROOT / "outputs" / "a4-32-local-hold" / RUN_ID
command = [
    sys.executable, "-u", "-m", "assignment.training.planning.evaluate_goal_actor",
    "--policy", str(ACTOR),
    "--a4-cumulative-step", "120000",
    "--world-model", str(WM),
    "--official-policy", str(OFFICIAL_ACTOR),
    "--goals", str(GOALS),
    "--scenes", str(SCENE),
    "--sample-seeds", str(SAMPLE_SEED),
    "--groups", "A4-32", "A4-32-HOLD", "A4-32-LOCAL-HOLD",
    "--horizon", "1795",
    "--workers", "1",
    "--gpus", "0",
    "--videos", "1",
    "--out", str(OUT),
]
print("running:", " ".join(command), flush=True)
subprocess.check_call(command, cwd=ROOT, env=dict(os.environ, PYTHONPATH=str(ROOT)))
print("completed:", OUT)


In [ ]:
import json
from IPython.display import JSON, Markdown, Video, display

summary = json.loads((OUT / "summary.json").read_text())
modes = ("A4-32", "A4-32-HOLD", "A4-32-LOCAL-HOLD")
table = "| Group | Peak | Final | Success | Four-box retention | Mean decision ms |\n|---|---:|---:|---:|---:|---:|\n"
episode_dirs = {}
for mode in modes:
    folder = OUT / mode / "episodes" / f"scene-{SCENE}.sample-{SAMPLE_SEED}"
    episode_dirs[mode] = folder
    episode = json.loads((folder / "episode.json").read_text())
    group = summary["groups"][mode]
    retention = group["target_retention"]
    retained = f"{retention['retained']}/{retention['at_risk']}" if retention["at_risk"] else "n/a"
    table += (f"| {mode} | {episode['peak_boxes']} | {episode['final_boxes']} | "
              f"{episode['success']} | {retained} | {1000 * group['mean_decision_s']:.1f} |\n")
display(Markdown("### Controlled LOCAL-HOLD results\n\n" + table + f"\nEvidence: `{OUT}`"))

audit = "| Group | Latched decisions | Entries | Exits | Safe eligible rate | Selected wrist RMS |\n|---|---:|---:|---:|---:|---:|\n"
for mode in modes:
    rows = [json.loads(line) for line in (episode_dirs[mode] / "steps.jsonl").read_text().splitlines()]
    active = [r for r in rows if r.get("hold_latched")]
    eligible_rows = [r for r in active if r.get("eligible") is not None]
    safe_rate = (sum(any(r["eligible"]) for r in eligible_rows) / len(eligible_rows)
                 if eligible_rows else None)
    selected_rms = [r["candidate_first_block_wrist_rms"][r["selected_candidate"]]
                    for r in active if r.get("candidate_first_block_wrist_rms") is not None]
    audit += (f"| {mode} | {len(active)} | {sum(r.get('hold_entered', False) for r in rows)} | "
              f"{sum(r.get('hold_exited', False) for r in rows)} | "
              f"{safe_rate if safe_rate is not None else 'n/a'} | "
              f"{sum(selected_rms)/len(selected_rms):.4f}" if selected_rms else
              f"| {mode} | {len(active)} | {sum(r.get('hold_entered', False) for r in rows)} | "
              f"{sum(r.get('hold_exited', False) for r in rows)} | "
              f"{safe_rate if safe_rate is not None else 'n/a'} | n/a")
    audit += " |\n"
display(Markdown("### HOLD audit\n\n" + audit))
for mode, folder in episode_dirs.items():
    display(Markdown(f"#### {mode}"))
    display(Video(str(folder / "video.mp4"), embed=True, width=512))
display(JSON(summary))
